In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
"""
Configuration for Math Reasoning Diffusion Model
Optimized for 2x T4 GPUs (16GB each)
"""

import torch
from dataclasses import dataclass, field
from typing import Optional, List

@dataclass
class ModelConfig:
    """Model architecture configuration"""
    # Embedding dimensions
    hidden_dim: int = 512  # Reduced from 768 for T4 memory
    intermediate_dim: int = 2048
    
    # Transformer architecture
    num_layers: int = 8
    num_heads: int = 8
    dropout: float = 0.1
    
    # Vocabulary (will use GPT-2 tokenizer)
    vocab_size: int = 50261  # GPT-2 vocabulary size
    max_seq_len: int = 512
    
    # Diffusion settings
    diffusion_type: str = "discrete"  # "discrete" (MDLM-style) or "continuous"
    timesteps: int = 1000
    noise_schedule: str = "cosine"  # "linear", "cosine", "sqrt"
    
    # Self-conditioning (improves quality significantly)
    use_self_conditioning: bool = True
    self_cond_prob: float = 0.5


@dataclass
class TrainingConfig:
    """Training configuration optimized for 2x T4"""
    # Batch sizes
    batch_size_per_gpu: int = 4
    gradient_accumulation_steps: int = 8
    # Effective batch = 4 * 2 GPUs * 8 accum = 64
    
    # Learning rate
    learning_rate: float = 1e-4
    min_learning_rate: float = 1e-6
    weight_decay: float = 0.01
    
    # Schedule
    warmup_steps: int = 1000
    max_steps: int = 100000
    
    # Optimization
    max_grad_norm: float = 1.0
    use_amp: bool = True  # Mixed precision
    use_gradient_checkpointing: bool = True
    
    # EMA for stable generation
    use_ema: bool = True
    ema_decay: float = 0.9999
    
    # Logging
    log_every: int = 50
    eval_every: int = 1000
    save_every: int = 5000
    
    # Paths
    output_dir: str = "./outputs"
    checkpoint_path: Optional[str] = None


@dataclass
class DataConfig:
    """Dataset configuration"""
    dataset_name: str = "gsm8k"  # "gsm8k", "math", "combined"
    max_question_len: int = 128
    max_answer_len: int = 384  # CoT can be long
    
    # Preprocessing
    num_workers: int = 4
    prefetch_factor: int = 2
    
    # Data augmentation
    use_augmentation: bool = False


@dataclass
class SamplingConfig:
    """Inference/sampling configuration"""
    num_steps: int = 1000  # More steps = better quality
    sampler: str = "ddpm"  # "ddpm", "ddim", "ddpm_cache"
    
    # Temperature for discrete diffusion
    temperature: float = 1.0
    top_k: Optional[int] = None
    top_p: Optional[float] = 0.95
    
    # For morphing (question -> answer)
    morph_strength: float = 0.0  # 0 = start from mask, 1 = start from noised question


@dataclass
class Config:
    """Master configuration"""
    model: ModelConfig = field(default_factory=ModelConfig)
    training: TrainingConfig = field(default_factory=TrainingConfig)
    data: DataConfig = field(default_factory=DataConfig)
    sampling: SamplingConfig = field(default_factory=SamplingConfig)
    
    # Hardware
    seed: int = 42
    device: str = "cuda"
    num_gpus: int = 2
    
    def __post_init__(self):
        # Auto-detect GPUs
        if torch.cuda.is_available():
            self.num_gpus = torch.cuda.device_count()
            print(f"Detected {self.num_gpus} GPU(s)")
        else:
            self.device = "cpu"
            self.num_gpus = 0
            self.training.use_amp = False
            print("No GPU detected, using CPU")


def get_config(preset: str = "default") -> Config:
    """Get configuration preset"""
    config = Config()
    
    if preset == "small":
        # Smaller model for faster iteration
        config.model.hidden_dim = 256
        config.model.num_layers = 4
        config.model.intermediate_dim = 1024
        config.training.max_steps = 50000
        
    elif preset == "debug":
        # Minimal config for debugging
        config.model.hidden_dim = 128
        config.model.num_layers = 2
        config.model.intermediate_dim = 512
        config.model.max_seq_len = 128
        config.training.batch_size_per_gpu = 2
        config.training.max_steps = 1000
        config.training.log_every = 10
        config.training.eval_every = 100
        config.sampling.num_steps = 100
        
    elif preset == "large":
        # Larger model (may need gradient checkpointing)
        config.model.hidden_dim = 768
        config.model.num_layers = 12
        config.model.intermediate_dim = 3072
        config.training.batch_size_per_gpu = 2
        config.training.gradient_accumulation_steps = 16
        
    return config


In [3]:
"""
Dataset Module for Math Reasoning Diffusion

Supports:
- GSM8K (grade school math with CoT)
- MATH (competition math with CoT)
- Combined datasets
- Data augmentation

Format: Question -> Chain of Thought -> Answer
"""

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer
from datasets import load_dataset, concatenate_datasets
from typing import Optional, Dict, List, Tuple
import re
import random


class MathReasoningDataset(Dataset):
    """
    Dataset for math reasoning with chain-of-thought
    
    Each sample contains:
    - question: The math problem
    - solution: Step-by-step reasoning (CoT)
    - answer: Final numerical answer
    
    The model learns to generate: solution + answer given question
    """
    
    def __init__(
        self,
        dataset_name: str = "gsm8k",
        split: str = "train",
        tokenizer: Optional[GPT2Tokenizer] = None,
        max_question_len: int = 128,
        max_answer_len: int = 384,
        use_augmentation: bool = False
    ):
        self.dataset_name = dataset_name
        self.split = split
        self.max_question_len = max_question_len
        self.max_answer_len = max_answer_len
        self.use_augmentation = use_augmentation
        
        # Initialize tokenizer
        if tokenizer is None:
            self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
            # Add special tokens
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.add_special_tokens({
                'additional_special_tokens': ['[QUESTION]', '[SOLUTION]', '[ANSWER]', '[MASK]']
            })
        else:
            self.tokenizer = tokenizer
        
        self.mask_token_id = self.tokenizer.convert_tokens_to_ids('[MASK]')
        if self.mask_token_id == self.tokenizer.unk_token_id:
            # Fallback if [MASK] wasn't added properly
            self.mask_token_id = self.tokenizer.eos_token_id
        
        # Load dataset
        self.data = self._load_dataset()
        print(f"Loaded {len(self.data)} samples from {dataset_name} ({split})")
        
    def _load_dataset(self) -> List[Dict]:
        """Load and preprocess dataset"""
        if self.dataset_name == "gsm8k":
            return self._load_gsm8k()
        elif self.dataset_name == "math":
            return self._load_math()
        elif self.dataset_name == "combined":
            gsm8k = self._load_gsm8k()
            math_data = self._load_math()
            return gsm8k + math_data
        else:
            raise ValueError(f"Unknown dataset: {self.dataset_name}")
    
    def _load_gsm8k(self) -> List[Dict]:
        """Load GSM8K dataset"""
        try:
            dataset = load_dataset("gsm8k", "main", split=self.split)
        except Exception as e:
            print(f"Error loading GSM8K: {e}")
            print("Using synthetic data for testing...")
            return self._create_synthetic_data(100)
        
        processed = []
        for item in dataset:
            question = item['question'].strip()
            answer_text = item['answer']
            
            # GSM8K format: reasoning with <<calc=result>> then #### final_answer
            # Extract the reasoning and final answer
            parts = answer_text.split('####')
            if len(parts) == 2:
                solution = parts[0].strip()
                final_answer = parts[1].strip()
            else:
                solution = answer_text
                final_answer = ""
            
            # Clean up the <<calculation>> format
            solution = re.sub(r'<<[^>]+>>', '', solution).strip()
            solution = ' '.join(solution.split())  # Normalize whitespace
            
            processed.append({
                'question': question,
                'solution': solution,
                'answer': final_answer
            })
        
        return processed
    
    def _load_math(self) -> List[Dict]:
        """Load MATH dataset"""
        try:
            dataset = load_dataset("qwedsacf/competition_math", split=self.split)
        except Exception as e:
            print(f"Error loading MATH dataset: {e}")
            return []
        
        processed = []
        for item in dataset:
            question = item['problem'].strip()
            solution = item['solution'].strip()
            
            # Extract answer from \boxed{}
            answer_match = re.search(r'\\boxed\{([^}]+)\}', solution)
            if answer_match:
                final_answer = answer_match.group(1)
            else:
                final_answer = ""
            
            # Clean LaTeX for simpler processing
            solution = self._clean_latex(solution)
            
            processed.append({
                'question': question,
                'solution': solution,
                'answer': final_answer
            })
        
        return processed
    
    def _clean_latex(self, text: str) -> str:
        """Clean LaTeX formatting for simpler tokenization"""
        # Remove \boxed{} but keep content
        text = re.sub(r'\\boxed\{([^}]+)\}', r'\1', text)
        # Simplify common LaTeX
        text = text.replace('\\frac', '/')
        text = text.replace('\\cdot', '*')
        text = text.replace('\\times', '*')
        text = re.sub(r'\$+', '', text)  # Remove dollar signs
        return text.strip()
    
    def _create_synthetic_data(self, n: int) -> List[Dict]:
        """Create synthetic math data for testing"""
        data = []
        operations = [
            ("addition", lambda a, b: (f"{a} + {b}", a + b)),
            ("subtraction", lambda a, b: (f"{a} - {b}", a - b)),
            ("multiplication", lambda a, b: (f"{a} × {b}", a * b)),
        ]
        
        templates = [
            "What is {expr}?",
            "Calculate {expr}.",
            "Find the result of {expr}.",
            "Compute {expr}.",
        ]
        
        solution_templates = [
            "Let me solve this step by step. We need to calculate {expr}. The result is {ans}.",
            "To find {expr}, I'll compute it directly. {a} and {b} gives us {ans}.",
            "Step 1: Identify the operation. Step 2: Calculate {expr} = {ans}.",
        ]
        
        for i in range(n):
            a = random.randint(1, 100)
            b = random.randint(1, 100)
            op_name, op_func = random.choice(operations)
            expr, result = op_func(a, b)
            
            question = random.choice(templates).format(expr=expr)
            solution = random.choice(solution_templates).format(
                expr=expr, a=a, b=b, ans=result
            )
            
            data.append({
                'question': question,
                'solution': solution,
                'answer': str(result)
            })
        
        return data
    
    def _augment(self, item: Dict) -> Dict:
        """Apply data augmentation"""
        if not self.use_augmentation:
            return item
        
        # Random number substitution (keep structure, change values)
        # This is a simple form of augmentation
        question = item['question']
        solution = item['solution']
        answer = item['answer']
        
        # 50% chance to paraphrase question slightly
        if random.random() < 0.5:
            prefixes = ["", "Question: ", "Problem: ", "Solve: "]
            question = random.choice(prefixes) + question
        
        return {
            'question': question,
            'solution': solution,
            'answer': answer
        }
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.data[idx]
        if self.use_augmentation:
            item = self._augment(item)
        
        # Format: [QUESTION] question [SOLUTION] solution [ANSWER] answer
        question_text = f"[QUESTION] {item['question']}"
        solution_text = f"[SOLUTION] {item['solution']} [ANSWER] {item['answer']}"
        
        # Tokenize question (condition)
        question_enc = self.tokenizer(
            question_text,
            max_length=self.max_question_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Tokenize solution (target)
        solution_enc = self.tokenizer(
            solution_text,
            max_length=self.max_answer_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'question_ids': question_enc['input_ids'].squeeze(0),
            'question_mask': question_enc['attention_mask'].squeeze(0),
            'solution_ids': solution_enc['input_ids'].squeeze(0),
            'solution_mask': solution_enc['attention_mask'].squeeze(0),
            # Keep raw text for debugging
            'question_text': item['question'],
            'solution_text': f"{item['solution']} Answer: {item['answer']}"
        }


def create_dataloaders(
    config,
    tokenizer: Optional[GPT2Tokenizer] = None
) -> Tuple[DataLoader, DataLoader, GPT2Tokenizer]:
    """Create train and eval dataloaders"""
    
    # Create datasets
    train_dataset = MathReasoningDataset(
        dataset_name=config.data.dataset_name,
        split="train",
        tokenizer=tokenizer,
        max_question_len=config.data.max_question_len,
        max_answer_len=config.data.max_answer_len,
        use_augmentation=config.data.use_augmentation
    )
    
    # Get tokenizer from dataset if not provided
    tokenizer = train_dataset.tokenizer
    
    # Try to load test split, fall back to train subset
    try:
        eval_dataset = MathReasoningDataset(
            dataset_name=config.data.dataset_name,
            split="test",
            tokenizer=tokenizer,
            max_question_len=config.data.max_question_len,
            max_answer_len=config.data.max_answer_len,
            use_augmentation=False
        )
    except Exception:
        print("No test split available, using subset of train for eval")
        # Use last 10% of train as eval
        eval_size = max(100, len(train_dataset) // 10)
        train_dataset.data = train_dataset.data[:-eval_size]
        eval_data = train_dataset.data[-eval_size:]
        
        eval_dataset = MathReasoningDataset(
            dataset_name=config.data.dataset_name,
            split="train",
            tokenizer=tokenizer,
            max_question_len=config.data.max_question_len,
            max_answer_len=config.data.max_answer_len,
            use_augmentation=False
        )
        eval_dataset.data = eval_data
    
    # Custom collate function to handle text fields
    def collate_fn(batch):
        result = {
            'question_ids': torch.stack([b['question_ids'] for b in batch]),
            'question_mask': torch.stack([b['question_mask'] for b in batch]),
            'solution_ids': torch.stack([b['solution_ids'] for b in batch]),
            'solution_mask': torch.stack([b['solution_mask'] for b in batch]),
        }
        # Keep text for debugging (only in eval)
        if 'question_text' in batch[0]:
            result['question_text'] = [b['question_text'] for b in batch]
            result['solution_text'] = [b['solution_text'] for b in batch]
        return result
    
    # Create dataloaders
    real_batch_size = config.training.batch_size_per_gpu * torch.cuda.device_count()
    train_loader = DataLoader(
        train_dataset,
        batch_size=real_batch_size,
        shuffle=True,
        num_workers=config.data.num_workers,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    eval_loader = DataLoader(
        eval_dataset,
        batch_size=real_batch_size,
        shuffle=False,
        num_workers=config.data.num_workers,
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    return train_loader, eval_loader, tokenizer


class MetaMathDataset(Dataset):
    """
    MetaMathQA dataset - high quality synthetic math data
    Much larger than GSM8K with good CoT annotations
    """
    
    def __init__(
        self,
        split: str = "train",
        tokenizer: Optional[GPT2Tokenizer] = None,
        max_question_len: int = 128,
        max_answer_len: int = 384,
        max_samples: Optional[int] = None
    ):
        self.max_question_len = max_question_len
        self.max_answer_len = max_answer_len
        
        # Initialize tokenizer
        if tokenizer is None:
            self.tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.add_special_tokens({
                'additional_special_tokens': ['[QUESTION]', '[SOLUTION]', '[ANSWER]', '[MASK]']
            })
        else:
            self.tokenizer = tokenizer
        
        self.mask_token_id = self.tokenizer.eos_token_id
        
        # Load MetaMathQA
        try:
            dataset = load_dataset("meta-math/MetaMathQA", split=split)
            if max_samples:
                dataset = dataset.select(range(min(max_samples, len(dataset))))
            self.data = self._process_metamath(dataset)
            print(f"Loaded {len(self.data)} samples from MetaMathQA")
        except Exception as e:
            print(f"Error loading MetaMathQA: {e}")
            self.data = []
    
    def _process_metamath(self, dataset) -> List[Dict]:
        """Process MetaMathQA format"""
        processed = []
        for item in dataset:
            question = item.get('query', item.get('question', ''))
            response = item.get('response', item.get('answer', ''))
            
            # Extract final answer if present
            answer_match = re.search(r'(?:answer is|=)\s*([0-9,.-]+)', response, re.IGNORECASE)
            final_answer = answer_match.group(1) if answer_match else ""
            
            processed.append({
                'question': question.strip(),
                'solution': response.strip(),
                'answer': final_answer
            })
        
        return processed
    
    def __len__(self) -> int:
        return len(self.data)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        item = self.data[idx]
        
        question_text = f"[QUESTION] {item['question']}"
        solution_text = f"[SOLUTION] {item['solution']} [ANSWER] {item['answer']}"
        
        question_enc = self.tokenizer(
            question_text,
            max_length=self.max_question_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        solution_enc = self.tokenizer(
            solution_text,
            max_length=self.max_answer_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'question_ids': question_enc['input_ids'].squeeze(0),
            'question_mask': question_enc['attention_mask'].squeeze(0),
            'solution_ids': solution_enc['input_ids'].squeeze(0),
            'solution_mask': solution_enc['attention_mask'].squeeze(0),
        }


In [4]:
"""
Discrete Diffusion Language Model for Math Reasoning
Based on MDLM (Masked Diffusion Language Models) architecture

Key innovations:
1. Discrete masked diffusion (more stable than continuous)
2. Bidirectional transformer (full attention, not causal)
3. Self-conditioning for better quality
4. Adaptive layer normalization for time conditioning
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from typing import Optional, Tuple, Dict
from dataclasses import dataclass


class SinusoidalPositionEmbeddings(nn.Module):
    """Sinusoidal embeddings for diffusion timesteps"""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        device = t.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = t[:, None].float() * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class AdaptiveLayerNorm(nn.Module):
    """
    Adaptive Layer Normalization (adaLN)
    Modulates layer norm with scale and shift from time embeddings
    Used in DiT (Diffusion Transformer) architecture
    """
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        self.gamma_proj = nn.Linear(hidden_dim, hidden_dim)
        self.beta_proj = nn.Linear(hidden_dim, hidden_dim)
        
    def forward(self, x: torch.Tensor, time_emb: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq, hidden)
        # time_emb: (batch, hidden)
        normalized = self.norm(x)
        gamma = self.gamma_proj(time_emb).unsqueeze(1)  # (batch, 1, hidden)
        beta = self.beta_proj(time_emb).unsqueeze(1)
        return gamma * normalized + beta


class TransformerBlock(nn.Module):
    """
    Transformer block with adaptive layer norm for time conditioning
    Uses full bidirectional attention (not causal)
    """
    def __init__(
        self, 
        hidden_dim: int, 
        num_heads: int, 
        intermediate_dim: int,
        dropout: float = 0.1,
        use_flash_attention: bool = True
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        # Attention
        self.q_proj = nn.Linear(hidden_dim, hidden_dim)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim)
        self.out_proj = nn.Linear(hidden_dim, hidden_dim)
        
        # FFN
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, intermediate_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(intermediate_dim, hidden_dim),
            nn.Dropout(dropout)
        )
        
        # Adaptive layer norms
        self.adaln1 = AdaptiveLayerNorm(hidden_dim)
        self.adaln2 = AdaptiveLayerNorm(hidden_dim)
        
        self.attn_dropout = nn.Dropout(dropout)
        self.use_flash = use_flash_attention and hasattr(F, 'scaled_dot_product_attention')
        
    def forward(
        self, 
        x: torch.Tensor, 
        time_emb: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
        condition: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        batch_size, seq_len, _ = x.shape
        
        # Pre-norm with time conditioning
        normed = self.adaln1(x, time_emb)
        
        # Self-attention (bidirectional)
        q = self.q_proj(normed).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(normed).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(normed).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        
        # Optional cross-attention with condition
        if condition is not None:
            cond_len = condition.shape[1]
            k_cond = self.k_proj(condition).view(batch_size, cond_len, self.num_heads, self.head_dim).transpose(1, 2)
            v_cond = self.v_proj(condition).view(batch_size, cond_len, self.num_heads, self.head_dim).transpose(1, 2)
            k = torch.cat([k, k_cond], dim=2)
            v = torch.cat([v, v_cond], dim=2)
        
        if self.use_flash:
            # Use Flash Attention if available
            attn_out = F.scaled_dot_product_attention(
                q, k, v, 
                attn_mask=None,  # No causal mask - full bidirectional
                dropout_p=self.attn_dropout.p if self.training else 0.0
            )
        else:
            # Standard attention
            scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            if attention_mask is not None:
                scores = scores.masked_fill(attention_mask == 0, float('-inf'))
            attn_weights = F.softmax(scores, dim=-1)
            attn_weights = self.attn_dropout(attn_weights)
            attn_out = torch.matmul(attn_weights, v)
        
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.hidden_dim)
        attn_out = self.out_proj(attn_out)
        
        # Residual connection
        x = x + attn_out
        
        # FFN with adaptive layer norm
        normed = self.adaln2(x, time_emb)
        x = x + self.ffn(normed)
        
        return x


class MathDiffusionTransformer(nn.Module):
    """
    Main diffusion transformer for math reasoning
    
    Architecture:
    - Token embeddings (learned, not frozen BERT)
    - Sinusoidal position embeddings
    - Time embedding MLP
    - Stack of transformer blocks with adaLN
    - Output projection to vocabulary
    """
    def __init__(
        self,
        vocab_size: int = 50261,
        hidden_dim: int = 512,
        num_layers: int = 8,
        num_heads: int = 8,
        intermediate_dim: int = 2048,
        max_seq_len: int = 512,
        dropout: float = 0.1,
        use_self_conditioning: bool = True
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.hidden_dim = hidden_dim
        self.max_seq_len = max_seq_len
        self.use_self_conditioning = use_self_conditioning
        
        # Token embeddings (learned from scratch)
        self.token_embedding = nn.Embedding(vocab_size, hidden_dim)
        self.position_embedding = nn.Embedding(max_seq_len, hidden_dim)
        
        # Time embedding
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(hidden_dim),
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim)
        )
        
        # Self-conditioning: project previous prediction
        if use_self_conditioning:
            self.self_cond_proj = nn.Linear(hidden_dim, hidden_dim)
        
        # Condition encoder (for question)
        self.condition_encoder = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        
        # Transformer blocks
        self.layers = nn.ModuleList([
            TransformerBlock(
                hidden_dim=hidden_dim,
                num_heads=num_heads,
                intermediate_dim=intermediate_dim,
                dropout=dropout
            )
            for _ in range(num_layers)
        ])
        
        # Final layer norm and output projection
        self.final_norm = nn.LayerNorm(hidden_dim)
        self.output_proj = nn.Linear(hidden_dim, vocab_size, bias=False)
        
        # Tie input and output embeddings
        self.output_proj.weight = self.token_embedding.weight
        
        # Initialize weights
        self._init_weights()



    def forward(
        self,
        input_ids: torch.Tensor,
        timesteps: torch.Tensor,
        condition_ids: Optional[torch.Tensor] = None,
        condition_mask: Optional[torch.Tensor] = None,
        self_cond: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Forward pass for denoising
        
        Args:
            input_ids: Noisy/masked token ids (batch, seq_len)
            timesteps: Diffusion timesteps (batch,)
            condition_ids: Question token ids (batch, cond_len)
            condition_mask: Attention mask for condition (batch, cond_len)
            self_cond: Previous prediction for self-conditioning (batch, seq_len, hidden)
            attention_mask: Attention mask for input (batch, seq_len)
            
        Returns:
            logits: Predicted token logits (batch, seq_len, vocab_size)
        """
        batch_size, seq_len = input_ids.shape
        device = input_ids.device
        
        # Ensure positions are within the valid range
        positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)
        positions = torch.clamp(positions, max=seq_len - 1)  # Ensure positions are within bounds
        
        # Ensure input_ids is within valid index range for embedding
        assert input_ids.max().item() < self.token_embedding.weight.size(0), f"Invalid index in input_ids: {input_ids.max().item()}"

        # Token + position embeddings
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        
        # Time embedding
        time_emb = self.time_mlp(timesteps)
        
        # Self-conditioning
        if self.use_self_conditioning and self_cond is not None:
            x = x + self.self_cond_proj(self_cond)
        
        # Encode condition (question)
        condition = None
        if condition_ids is not None:
            cond_positions = torch.arange(condition_ids.shape[1], device=device).unsqueeze(0)
            cond_emb = self.token_embedding(condition_ids) + self.position_embedding(cond_positions)
            condition = self.condition_encoder(cond_emb)
        
        # Transformer layers
        for layer in self.layers:
            x = layer(x, time_emb, attention_mask, condition)
        
        # Output projection
        x = self.final_norm(x)
        logits = self.output_proj(x)
        
        return logits

    def _init_weights(self):
        """Initialize weights with small values for stable training"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                
    def forward(
        self,
        input_ids: torch.Tensor,
        timesteps: torch.Tensor,
        condition_ids: Optional[torch.Tensor] = None,
        condition_mask: Optional[torch.Tensor] = None,
        self_cond: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """
        Forward pass for denoising
        
        Args:
            input_ids: Noisy/masked token ids (batch, seq_len)
            timesteps: Diffusion timesteps (batch,)
            condition_ids: Question token ids (batch, cond_len)
            condition_mask: Attention mask for condition (batch, cond_len)
            self_cond: Previous prediction for self-conditioning (batch, seq_len, hidden)
            attention_mask: Attention mask for input (batch, seq_len)
            
        Returns:
            logits: Predicted token logits (batch, seq_len, vocab_size)
        """
        batch_size, seq_len = input_ids.shape
        device = input_ids.device
        
        # Token + position embeddings
        positions = torch.arange(seq_len, device=device).unsqueeze(0).expand(batch_size, -1)
        x = self.token_embedding(input_ids) + self.position_embedding(positions)
        
        # Time embedding
        time_emb = self.time_mlp(timesteps)
        
        # Self-conditioning
        if self.use_self_conditioning and self_cond is not None:
            x = x + self.self_cond_proj(self_cond)
        
        # Encode condition (question)
        condition = None
        if condition_ids is not None:
            cond_positions = torch.arange(condition_ids.shape[1], device=device).unsqueeze(0)
            cond_emb = self.token_embedding(condition_ids) + self.position_embedding(cond_positions)
            condition = self.condition_encoder(cond_emb)
        
        # Transformer layers
        for layer in self.layers:
            x = layer(x, time_emb, attention_mask, condition)
        
        # Output projection
        x = self.final_norm(x)
        logits = self.output_proj(x)
        
        return logits


class DiscreteDiffusion(nn.Module):
    """
    Discrete Diffusion for Math Reasoning
    
    Uses masked diffusion (MDLM-style):
    - Forward process: gradually mask tokens
    - Reverse process: predict and unmask tokens
    
    This is more stable than continuous diffusion for text
    """
    def __init__(
        self,
        model: MathDiffusionTransformer,
        vocab_size: int = 50261,
        timesteps: int = 1000,
        mask_token_id: int = 50256,  # Use GPT-2 <|endoftext|> or special mask
        noise_schedule: str = "cosine"
    ):
        super().__init__()
        self.model = model
        self.vocab_size = vocab_size
        self.timesteps = timesteps
        self.mask_token_id = mask_token_id
        
        # Compute noise schedule (probability of masking at each timestep)
        self.register_buffer('mask_schedule', self._compute_schedule(noise_schedule))

    def forward(
        self, 
        input_ids: torch.Tensor,  # Noisy/masked token ids (batch, seq_len)
        timesteps: torch.Tensor,  # Diffusion timesteps (batch,)
        condition_ids: Optional[torch.Tensor] = None,  # Question token ids (batch, cond_len)
        attention_mask: Optional[torch.Tensor] = None  # Attention mask for input
    ) -> torch.Tensor:
        """
        Forward pass for denoising
        
        Args:
            input_ids: Noisy/masked token ids (batch, seq_len)
            timesteps: Diffusion timesteps (batch,)
            condition_ids: Question token ids (batch, cond_len)
            attention_mask: Attention mask for input (batch, seq_len)
            
        Returns:
            logits: Predicted token logits (batch, seq_len, vocab_size)
        """
        # Call the model (MathDiffusionTransformer) for the forward pass
        logits = self.model(
            input_ids=input_ids,
            timesteps=timesteps,
            condition_ids=condition_ids,
            attention_mask=attention_mask
        )
        return logits

    def q_sample(
        self, 
        x_0: torch.Tensor, 
        t: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward diffusion: mask tokens according to schedule
        
        Args:
            x_0: Clean token ids (batch, seq_len)
            t: Timesteps (batch,)
            
        Returns:
            x_t: Masked token ids
            mask: Boolean mask of which tokens are masked
        """
        batch_size, seq_len = x_0.shape
        device = x_0.device
        
        # Get mask probability for each sample
        mask_prob = self.mask_schedule[t].unsqueeze(1)  # (batch, 1)
        
        # Sample which tokens to mask
        rand = torch.rand(batch_size, seq_len, device=device)
        mask = rand < mask_prob  # True = masked
        
        # Apply masking
        x_t = x_0.clone()
        x_t[mask] = self.mask_token_id
        
        return x_t, mask



          
        
    def _compute_schedule(self, schedule_type: str) -> torch.Tensor:
        """Compute masking probability schedule"""
        t = torch.linspace(0, 1, self.timesteps + 1)
        
        if schedule_type == "linear":
            mask_prob = t
        elif schedule_type == "cosine":
            # Cosine schedule (slower at extremes)
            mask_prob = 1 - torch.cos(t * math.pi / 2)
        elif schedule_type == "sqrt":
            mask_prob = torch.sqrt(t)
        else:
            raise ValueError(f"Unknown schedule: {schedule_type}")
        
        return mask_prob
    
    def q_sample(
        self, 
        x_0: torch.Tensor, 
        t: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward diffusion: mask tokens according to schedule
        
        Args:
            x_0: Clean token ids (batch, seq_len)
            t: Timesteps (batch,)
            
        Returns:
            x_t: Masked token ids
            mask: Boolean mask of which tokens are masked
        """
        batch_size, seq_len = x_0.shape
        device = x_0.device
        
        # Get mask probability for each sample
        mask_prob = self.mask_schedule[t].unsqueeze(1)  # (batch, 1)
        
        # Sample which tokens to mask
        rand = torch.rand(batch_size, seq_len, device=device)
        mask = rand < mask_prob  # True = masked
        
        # Apply masking
        x_t = x_0.clone()
        x_t[mask] = self.mask_token_id
        
        return x_t, mask
    
    def compute_loss(
        self,
        x_0: torch.Tensor,
        condition_ids: Optional[torch.Tensor] = None,
        condition_mask: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        self_cond_prob: float = 0.5
    ) -> Dict[str, torch.Tensor]:
        """
        Compute training loss
        
        Uses cross-entropy on masked positions (MDLM-style)
        """
        batch_size, seq_len = x_0.shape
        device = x_0.device
        
        # Sample random timesteps
        t = torch.randint(1, self.timesteps + 1, (batch_size,), device=device)
        
        # Forward diffusion (mask tokens)
        x_t, mask = self.q_sample(x_0, t)
        
        # Self-conditioning: sometimes predict without previous output
        self_cond = None
        if self.model.use_self_conditioning and torch.rand(1).item() < self_cond_prob:
            with torch.no_grad():
                # Get previous prediction for self-conditioning
                prev_logits = self.model(
                    x_t, t, condition_ids, condition_mask, None, attention_mask
                )
                # Convert to embeddings
                prev_probs = F.softmax(prev_logits, dim=-1)
                self_cond = torch.matmul(prev_probs, self.model.token_embedding.weight)
        
        # Predict clean tokens
        logits = self.model(x_t, t, condition_ids, condition_mask, self_cond, attention_mask)
        
        # Compute loss only on masked positions
        # Flatten for cross-entropy
        logits_flat = logits.view(-1, self.vocab_size)
        targets_flat = x_0.view(-1)
        mask_flat = mask.view(-1)
        
        # Cross-entropy on masked positions
        loss_all = F.cross_entropy(logits_flat, targets_flat, reduction='none')
        loss_masked = (loss_all * mask_flat.float()).sum() / (mask_flat.sum() + 1e-8)
        
        # Also compute accuracy on masked positions
        with torch.no_grad():
            preds = logits.argmax(dim=-1).view(-1)
            acc = ((preds == targets_flat) * mask_flat.float()).sum() / (mask_flat.sum() + 1e-8)
        
        return {
            'loss': loss_masked,
            'accuracy': acc,
            'num_masked': mask_flat.sum()
        }
    
    @torch.no_grad()
    def sample(
        self,
        condition_ids: torch.Tensor,
        condition_mask: Optional[torch.Tensor] = None,
        seq_len: int = 384,
        num_steps: Optional[int] = None,
        temperature: float = 1.0,
        top_k: Optional[int] = None,
        top_p: Optional[float] = 0.95,
        verbose: bool = False
    ) -> Tuple[torch.Tensor, list]:
        """
        Sample from the model using reverse diffusion
        
        Args:
            condition_ids: Question token ids (batch, cond_len)
            condition_mask: Attention mask for condition
            seq_len: Length of sequence to generate
            num_steps: Number of sampling steps (default: self.timesteps)
            temperature: Sampling temperature
            top_k: Top-k filtering
            top_p: Nucleus sampling threshold
            verbose: Print progress
            
        Returns:
            samples: Generated token ids (batch, seq_len)
            history: List of intermediate states
        """
        self.model.eval()
        device = condition_ids.device
        batch_size = condition_ids.shape[0]
        num_steps = num_steps or self.timesteps
        
        # Start from fully masked sequence
        x = torch.full((batch_size, seq_len), self.mask_token_id, device=device)
        
        # Track history
        history = []
        self_cond = None
        
        # Reverse diffusion
        step_size = self.timesteps // num_steps
        timesteps = list(range(self.timesteps, 0, -step_size))
        
        if verbose:
            from tqdm import tqdm
            timesteps = tqdm(timesteps, desc="Sampling")
        
        for t in timesteps:
            t_tensor = torch.full((batch_size,), t, device=device, dtype=torch.long)
            
            # Predict clean tokens
            logits = self.model(x, t_tensor, condition_ids, condition_mask, self_cond)
            
            # Apply temperature
            logits = logits / temperature
            
            # Top-k filtering
            if top_k is not None:
                indices_to_remove = logits < torch.topk(logits, top_k, dim=-1)[0][..., -1, None]
                logits[indices_to_remove] = float('-inf')
            
            # Top-p (nucleus) filtering
            if top_p is not None:
                sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
                cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0
                indices_to_remove = sorted_indices_to_remove.scatter(-1, sorted_indices, sorted_indices_to_remove)
                logits[indices_to_remove] = float('-inf')
            
            # Sample from distribution
            probs = F.softmax(logits, dim=-1)
            samples = torch.multinomial(probs.view(-1, self.vocab_size), 1).view(batch_size, seq_len)
            
            # Only update masked positions
            is_masked = (x == self.mask_token_id)
            
            # Compute which positions to unmask at this step
            # More positions get unmasked as t decreases
            current_mask_prob = self.mask_schedule[t]
            next_mask_prob = self.mask_schedule[max(0, t - step_size)]
            
            # Unmask some positions
            unmask_prob = (current_mask_prob - next_mask_prob) / (current_mask_prob + 1e-8)
            unmask = torch.rand_like(x.float()) < unmask_prob
            update_mask = is_masked & unmask
            
            x = torch.where(update_mask, samples, x)
            
            # Update self-conditioning
            if self.model.use_self_conditioning:
                self_cond = torch.matmul(probs, self.model.token_embedding.weight)
            
            # Save to history periodically
            if t % 100 == 0:
                history.append(x.clone())
        
        # Final prediction for any remaining masked tokens
        t_tensor = torch.ones((batch_size,), device=device, dtype=torch.long)
        logits = self.model(x, t_tensor, condition_ids, condition_mask, self_cond)
        final_samples = logits.argmax(dim=-1)
        
        # Replace any remaining masks
        is_masked = (x == self.mask_token_id)
        x = torch.where(is_masked, final_samples, x)
        
        return x, history


def create_model(config) -> DiscreteDiffusion:
    """Create model from config"""
    transformer = MathDiffusionTransformer(
        vocab_size=config.model.vocab_size,
        hidden_dim=config.model.hidden_dim,
        num_layers=config.model.num_layers,
        num_heads=config.model.num_heads,
        intermediate_dim=config.model.intermediate_dim,
        max_seq_len=config.model.max_seq_len,
        dropout=config.model.dropout,
        use_self_conditioning=config.model.use_self_conditioning
    )
    
    diffusion = DiscreteDiffusion(
        model=transformer,
        vocab_size=config.model.vocab_size,
        timesteps=config.model.timesteps,
        noise_schedule=config.model.noise_schedule
    )
    
    return diffusion


In [5]:
#!/usr/bin/env python3
"""
Math Reasoning Diffusion Model - Main Entry Point

A discrete diffusion language model for mathematical reasoning
with chain-of-thought generation.

Usage:
    # Train on GSM8K with default settings (2x T4 GPUs)
    python main.py train --dataset gsm8k
    
    # Train with custom settings
    python main.py train --config small --max_steps 50000
    
    # Interactive inference
    python main.py infer --checkpoint outputs/checkpoint_final.pt
    
    # Batch inference
    python main.py infer --checkpoint outputs/checkpoint_final.pt --mode batch --input questions.json
    
    # Quick test
    python main.py test
"""

import os
import sys
import argparse


def train_command(args):
    """Run training"""
    # from train import main as train
    
    # Build sys.argv for train.py
    train_args = ['train.py']
    train_args.extend(['--config', args.config])
    train_args.extend(['--output_dir', args.output_dir])
    train_args.extend(['--dataset', args.dataset])
    
    if args.checkpoint:
        train_args.extend(['--checkpoint', args.checkpoint])
    if args.max_steps:
        train_args.extend(['--max_steps', str(args.max_steps)])
    if args.batch_size:
        train_args.extend(['--batch_size', str(args.batch_size)])
    
    sys.argv = train_args
    train()


def infer_command(args):
    """Run inference"""
    # from inference import main as infer_main
    
    # Build sys.argv for inference.py
    infer_args = ['inference.py']
    infer_args.extend(['--checkpoint', args.checkpoint])
    infer_args.extend(['--mode', args.mode])
    
    if args.input:
        infer_args.extend(['--input', args.input])
    if args.output:
        infer_args.extend(['--output', args.output])
    if args.question:
        infer_args.extend(['--question', args.question])
    if args.num_steps:
        infer_args.extend(['--num_steps', str(args.num_steps)])
    
    infer_args.extend(['--temperature', str(args.temperature)])
    infer_args.extend(['--top_p', str(args.top_p)])
    infer_args.extend(['--device', args.device])
    
    sys.argv = infer_args
    infer_main()


def test_command(args):
    """Run quick test to verify setup"""
    print("="*60)
    print("Running Quick Test")
    print("="*60)
    
    import torch
    print(f"\nPyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA devices: {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
            mem = torch.cuda.get_device_properties(i).total_memory / 1e9
            print(f"         Memory: {mem:.1f} GB")
    
    print("\n" + "-"*40)
    print("Testing imports...")
    
    try:
        from transformers import GPT2Tokenizer
        print("✓ transformers")
    except ImportError as e:
        print(f"✗ transformers: {e}")
    
    try:
        from datasets import load_dataset
        print("✓ datasets")
    except ImportError as e:
        print(f"✗ datasets: {e}")
    
    try:
        from config import get_config
        print("✓ config")
    except ImportError as e:
        print(f"✗ config: {e}")
    
    try:
        from model import create_model
        print("✓ model")
    except ImportError as e:
        print(f"✗ model: {e}")
    
    try:
        from dataset import MathReasoningDataset
        print("✓ dataset")
    except ImportError as e:
        print(f"✗ dataset: {e}")
    
    print("\n" + "-"*40)
    print("Testing model creation...")
    
    try:
        # from config import get_config
        # from model import create_model
        
        config = get_config('debug')  # Small config for quick test
        model = create_model(config)
        
        num_params = sum(p.numel() for p in model.parameters())
        print(f"✓ Model created: {num_params:,} parameters")
        
        # Test forward pass
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = model.to(config.device)
        
        # 🚀 ENABLE MULTI-GPU
        if torch.cuda.device_count() > 1:
            print(f"🔥 Boosting training with {torch.cuda.device_count()} GPUs!")
            model = torch.nn.DataParallel(model)        
        batch_size = 2
        seq_len = 64
        cond_len = 32
        
        x = torch.randint(0, 1000, (batch_size, seq_len), device=device)
        t = torch.randint(1, 100, (batch_size,), device=device)
        cond = torch.randint(0, 1000, (batch_size, cond_len), device=device)
        
        with torch.no_grad():
            loss_dict = model.compute_loss(x, condition_ids=cond)
        
        print(f"✓ Forward pass: loss = {loss_dict['loss'].item():.4f}")
        
        # Test sampling
        samples, _ = model.sample(
            condition_ids=cond,
            seq_len=32,
            num_steps=10,
            verbose=False
        )
        print(f"✓ Sampling: shape = {samples.shape}")
        
    except Exception as e:
        print(f"✗ Model test failed: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "-"*40)
    print("Testing dataset loading...")
    
    try:
        # from dataset import MathReasoningDataset
        # from transformers import GPT2Tokenizer
        
        tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
        tokenizer.pad_token = tokenizer.eos_token
        
        # Try synthetic data first (always works)
        dataset = MathReasoningDataset(
            dataset_name='gsm8k',
            split='train',
            tokenizer=tokenizer,
            max_question_len=64,
            max_answer_len=128
        )
        
        sample = dataset[0]
        print(f"✓ Dataset loaded: {len(dataset)} samples")
        print(f"  Question shape: {sample['question_ids'].shape}")
        print(f"  Solution shape: {sample['solution_ids'].shape}")
        
    except Exception as e:
        print(f"✗ Dataset test failed: {e}")
        import traceback
        traceback.print_exc()
    
    print("\n" + "="*60)
    print("Test Complete!")
    print("="*60)
    
    print("\nQuick Start:")
    print("  Train:  python main.py train --config debug --max_steps 1000")
    print("  Infer:  python main.py infer --checkpoint outputs/checkpoint_1000.pt")



In [6]:
!pip install --upgrade torch torchvision torchaudio


In [7]:
class DistributionWrapper(nn.Module):
    """
    Wraps the model so DataParallel can split the 'compute_loss' function across GPUs.
    """
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, solution_ids, condition_ids, condition_mask, attention_mask, self_cond_prob):
        # This forward pass now triggers compute_loss on EACH GPU
        return self.model.compute_loss(
            solution_ids,
            condition_ids=condition_ids,
            condition_mask=condition_mask,
            attention_mask=attention_mask,
            self_cond_prob=self_cond_prob
        )
    
    # Allow access to original methods (like sample)
    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.model, name)

In [8]:
import os
import sys
import math
import time
import argparse
from pathlib import Path
from typing import Optional, Dict
from contextlib import nullcontext
import warnings

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data.distributed import DistributedSampler
from torch.utils.data import DataLoader
from torch.amp import autocast, GradScaler
from tqdm import tqdm

warnings.filterwarnings("ignore", message="Was asked to gather along dimension 0")

# --- WRAPPER ---
class DistributionWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, solution_ids, condition_ids, condition_mask, attention_mask, self_cond_prob):
        return self.model.compute_loss(solution_ids, condition_ids, condition_mask, attention_mask, self_cond_prob)
    def __getattr__(self, name):
        try: return super().__getattr__(name)
        except AttributeError: return getattr(self.model, name)

# --- UTILITIES ---
class EMA:
    def __init__(self, model: nn.Module, decay: float = 0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        raw_model = model.module if hasattr(model, 'module') else model
        for name, param in raw_model.named_parameters():
            if param.requires_grad: self.shadow[name] = param.data.clone()
    def update(self):
        raw_model = self.model.module if hasattr(self.model, 'module') else self.model
        for name, param in raw_model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = (self.decay * self.shadow[name] + (1 - self.decay) * param.data)
    def apply_shadow(self): pass # skipped for brevity
    def restore(self): pass

def get_lr_scheduler(optimizer, config, num_training_steps: int):
    from torch.optim.lr_scheduler import LambdaLR
    warmup_steps = config.training.warmup_steps
    def lr_lambda(current_step):
        if current_step < warmup_steps: return float(current_step) / float(max(1, warmup_steps))
        else:
            progress = float(current_step - warmup_steps) / float(max(1, num_training_steps - warmup_steps))
            return max(config.training.min_learning_rate / config.training.learning_rate, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)

def save_checkpoint(model, optimizer, scheduler, scaler, ema, step, config, path):
    if hasattr(model, 'module'):
        inner = model.module
        raw_model = inner.model if hasattr(inner, 'model') else inner
    elif hasattr(model, 'model'): raw_model = model.model
    else: raw_model = model
    torch.save({'step': step, 'model_state_dict': raw_model.state_dict(), 'optimizer_state_dict': optimizer.state_dict(), 'config': config}, path)
    print(f"Saved checkpoint to {path}")

# --- MAIN TRAIN ---
def train(config):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Training on device: {device}")
    torch.manual_seed(config.seed)
    
    output_dir = Path(config.training.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    
    train_loader, eval_loader, tokenizer = create_dataloaders(config)
    
    # 1. Fix Vocab Size
    if tokenizer: config.model.vocab_size = len(tokenizer)
    elif config.model.vocab_size == 50257: config.model.vocab_size = 50261

    # 2. Create Model
    raw_model = create_model(config)
    
    # 3. RESIZE TOKEN EMBEDDINGS (Safe Vocab)
    if hasattr(raw_model, 'token_embedding'):
        curr_vocab = raw_model.token_embedding.weight.shape[0]
        if curr_vocab != config.model.vocab_size:
            print(f"Resizing Tokens {curr_vocab} -> {config.model.vocab_size}")
            raw_model.token_embedding = torch.nn.Embedding(config.model.vocab_size, config.model.hidden_dim)
            
    # 4. RESIZE POSITION EMBEDDINGS (Safe Length) - CRITICAL FIX
    # Ensure model handles at least 1024 or 2048 tokens to be safe
    SAFE_SEQ_LEN = 2048 
    if hasattr(raw_model, 'position_embedding'):
        curr_pos = raw_model.position_embedding.weight.shape[0]
        if curr_pos < SAFE_SEQ_LEN:
            print(f"Resizing Positions {curr_pos} -> {SAFE_SEQ_LEN}")
            raw_model.position_embedding = torch.nn.Embedding(SAFE_SEQ_LEN, config.model.hidden_dim)
            config.model.max_seq_len = SAFE_SEQ_LEN # Update config too

    raw_model = raw_model.to(device)
    model = DistributionWrapper(raw_model)

    if torch.cuda.device_count() > 1:
        print(f"🔥 ACTUALLY Boosting with {torch.cuda.device_count()} GPUs!")
        model = torch.nn.DataParallel(model)
        total_batch_size = config.training.batch_size_per_gpu * torch.cuda.device_count()
        train_loader = DataLoader(train_loader.dataset, batch_size=total_batch_size, shuffle=True, num_workers=config.data.num_workers, collate_fn=train_loader.collate_fn)
    
    if config.training.use_gradient_checkpointing:
        if hasattr(raw_model, 'model') and hasattr(raw_model.model, 'layers'):
             for layer in raw_model.model.layers: layer.gradient_checkpointing = True

    optimizer = torch.optim.AdamW(model.parameters(), lr=config.training.learning_rate)
    scheduler = get_lr_scheduler(optimizer, config, config.training.max_steps)
    scaler = GradScaler('cuda') if config.training.use_amp else None
    ema = EMA(model, decay=config.training.ema_decay) if config.training.use_ema else None

    model.train()
    step = 0
    pbar = tqdm(total=config.training.max_steps, desc="Training")
    
    MAX_ID = config.model.vocab_size - 1
    MAX_LEN = getattr(config.model, 'max_seq_len', 2048)

    while step < config.training.max_steps:
        for batch in train_loader:
            if step >= config.training.max_steps: break

            q_ids = batch['question_ids'].to(device)
            q_mask = batch['question_mask'].to(device)
            s_ids = batch['solution_ids'].to(device)
            s_mask = batch['solution_mask'].to(device)
            
            # --- 🔥 FINAL SANITIZATION BLOCK ---
            # 1. Truncate Length (Prevents Position Embedding Crash)
            if q_ids.size(1) > MAX_LEN:
                q_ids = q_ids[:, :MAX_LEN]
                q_mask = q_mask[:, :MAX_LEN]
            if s_ids.size(1) > MAX_LEN:
                s_ids = s_ids[:, :MAX_LEN]
                s_mask = s_mask[:, :MAX_LEN]

            # 2. Clamp Values (Prevents Token Embedding Crash)
            clean_q_ids = torch.clamp(q_ids, min=0, max=MAX_ID)
            clean_s_ids = torch.clamp(s_ids, min=0, max=MAX_ID)
            # -----------------------------------
            
            optimizer.zero_grad()
            with autocast('cuda', enabled=config.training.use_amp):
                metrics = model(solution_ids=clean_s_ids, condition_ids=clean_q_ids, condition_mask=q_mask, attention_mask=s_mask, self_cond_prob=config.model.self_cond_prob)
                loss = metrics['loss'].mean() / config.training.gradient_accumulation_steps
            
            if scaler: scaler.scale(loss).backward()
            else: loss.backward()
                
            if (step + 1) % config.training.gradient_accumulation_steps == 0:
                if scaler:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                
                scheduler.step()
                if ema: ema.update()
                step += 1
                pbar.update(1)
                
                if step % config.training.log_every == 0:
                    pbar.set_postfix({'loss': f"{loss.item() * config.training.gradient_accumulation_steps:.3f}"})
                
                if step % config.training.save_every == 0:
                    save_checkpoint(model, optimizer, scheduler, scaler, ema, step, config, str(output_dir / f'checkpoint_{step}.pt'))
    pbar.close()
    print("Training Complete!")

def main(args_list=None):
    parser = argparse.ArgumentParser()
    parser.add_argument('--config', type=str, default='default')
    parser.add_argument('--batch_size', type=int, default=None)
    parser.add_argument('--max_steps', type=int, default=None)
    parser.add_argument('--dataset', type=str, default='gsm8k')
    if args_list: args = parser.parse_args(args_list)
    else: args, _ = parser.parse_known_args()
    config = get_config(args.config)
    if args.batch_size: config.training.batch_size_per_gpu = args.batch_size
    if args.max_steps: config.training.max_steps = args.max_steps
    config.data.dataset_name = args.dataset
    train(config)

# main(['--config', 'debug', '--batch_size', '24'])

In [9]:
import argparse
import sys

def main(args_list=None):
    parser = argparse.ArgumentParser(
        description='Train Math Reasoning Diffusion Model',
        formatter_class=argparse.ArgumentDefaultsHelpFormatter
    )
    
    # 1. Define your arguments
    parser.add_argument('--config', type=str, default='debug',
                        choices=['default', 'small', 'debug', 'large'],
                        help='Configuration preset')
    parser.add_argument('--output_dir', type=str, default='./outputs',
                        help='Output directory')
    parser.add_argument('--checkpoint', type=str, default=None,
                        help='Resume from checkpoint')
    parser.add_argument('--max_steps', type=int, default=None,
                        help='Override max training steps')
    parser.add_argument('--batch_size', type=int, default=96,
                        help='Override batch size per GPU')
    parser.add_argument('--dataset', type=str, default='combined',
                        choices=['gsm8k', 'math', 'combined'],
                        help='Dataset to use')
    
    # 2. THE FIX: Handle Notebook Arguments
    if args_list is not None:
        # If you pass arguments manually: main(['--config', 'debug'])
        args = parser.parse_args(args_list)
    else:
        # If you run main() empty: It ignores the hidden '-f' flag from Colab
        args, unknown = parser.parse_known_args()

    # 3. Load Config
    print(f"Loading config: {args.config}")
    config = get_config(args.config)
    
    # 4. Apply Overrides
    config.training.output_dir = args.output_dir
    if args.checkpoint:
        config.training.checkpoint_path = args.checkpoint
    if args.max_steps:
        config.training.max_steps = args.max_steps
    if args.batch_size:
        config.training.batch_size_per_gpu = args.batch_size
    config.data.dataset_name = args.dataset
    
    # 5. SAFETY PATCH: Ensure vocab size matches data
    # (Prevents the CUDA error from earlier)
    if hasattr(config, 'model') and config.model.vocab_size == 50257:
        print("PATCH: Updating vocab_size to 50261 to match tokenizer")
        config.model.vocab_size = 50261

    # 6. Train
    print("Starting training...")
    train(config)

In [ ]:
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
# Run your main function after setting this
main()

Loading config: debug
Detected 2 GPU(s)
Starting training...
Training on device: cuda
Loaded 19973 samples from combined (train)
Error loading MATH dataset: Unknown split "test". Should be one of ['train'].
Loaded 1319 samples from combined (test)
🔥 ACTUALLY Boosting with 2 GPUs!


Training:   0%|          | 0/1000 [00:00<?, ?it/s]

In [21]:
# ---------------------------------------------------------
# PASTE THIS RIGHT BEFORE YOUR 'train()' CALL
# ---------------------------------------------------------

def debug_vocab_mismatch(model, tokenizer=None):
    print("--- DEBUGGING VOCAB SIZES ---")
    
    # 1. Check Model Embeddings
    if hasattr(model, 'get_input_embeddings'):
        embeddings = model.get_input_embeddings()
    elif hasattr(model, 'embedding'):
        embeddings = model.embedding
    else:
        # Fallback for simple models
        embeddings = getattr(model, 'embed_tokens', None)
    
    if embeddings is not None:
        model_vocab_size = embeddings.weight.shape[0]
        print(f"Model Embedding Size: {model_vocab_size}")
    else:
        print("Could not locate embedding layer automatically.")
        model_vocab_size = 0

    # 2. Check Configuration
    config_vocab = getattr(model.config, 'vocab_size', 'Not Found')
    print(f"Config Vocab Size:    {config_vocab}")
    
    # 3. Check Tokenizer (if you have one loaded)
    if tokenizer:
        print(f"Tokenizer Vocab Size: {len(tokenizer)}")
        
        # CRITICAL TEST: unexpected tokens
        test_tokens = tokenizer("7+5=12")['input_ids']
        print(f"Sample Token IDs: {test_tokens}")
        if max(test_tokens) >= model_vocab_size:
            print("\n[!!!!] CRITICAL ERROR DETECTED [!!!!]")
            print(f"Tokenizer produced ID {max(test_tokens)} which is >= Model Size {model_vocab_size}")
            print("SOLUTION: You must resize the model embeddings:")
            print("model.resize_token_embeddings(len(tokenizer))")
        else:
            print("\n[OK] Tokenizer IDs seem to fit in model.")

# Call this if you have your objects ready
debug_vocab_mismatch(model, tokenizer)

NameError: name 'model' is not defined